# Capa Silver - Transacciones de Venta

In [1]:
from spark_session import *
from pyspark.sql.functions import col
from pyspark.sql.types import DecimalType

## SparkSession

In [2]:
spark = create_spark_session(
    "retail-dlh-silver",
    configs=[
        configure_delta_extensions,
        configure_delta_catalog,
        configure_s3
    ]
)

## Extracción

In [3]:
# - Obtiene dataframe de la capa landing
df_LanSalTra = spark.table("landing.sales_transactions")

# - Obtiene dataframe de la capa bronze
df_BroSalTra = spark.table("bronze.sales_transaction")

# - Obtiene dataframe de la capa silver
df_SilProducts = spark.table("silver.products")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


## Transformación

In [4]:
# - Obtiene las distintas fechas de transacción de la capa landing
df_LanSalTra = df_LanSalTra.select(col("TRANSACTION_DATE")) \
    .distinct()

In [5]:
# - Obtiene transacciones de venta de la capa bronze en base a las distintas fechas de la capa landing
# - Obtiene el costo del producto
df_BroSalTra = df_BroSalTra.alias("bst") \
    .join(df_LanSalTra.alias("lst"), \
        (col("bst.TRANSACTION_DATE") == col("lst.TRANSACTION_DATE")),"inner") \
    .join(df_SilProducts.alias("sp"), \
        (col("bst.PRODUCT") == col("sp.PRODUCT_ID")), "inner")

In [6]:
# - Ajusta esquema
df_BroSalTra = df_BroSalTra.select(col("bst.TRANSACTION_DATE_ID").alias("TRANSACTION_DATE_ID")
    ,col("bst.CUST_NO").alias("CUST_ID")
    ,col("bst.PRODUCT").alias("PRODUCT_ID")
    ,col("bst.DEALERSHIP").alias("DEALERSHIP_ID")
    ,col("bst.PAYMENT_DESC").alias("PAYMENT_DESC")
    ,col("bst.PROMO_ID").alias("PROMO_ID")
    ,col("bst.DATE_ID").alias("DATE_ID")
    ,col("bst.TRANSACTION_DATE").alias("TRANSACTION_DATE")
    ,col("bst.TRANSACTION_ID").alias("TRANSACTION_ID")
    ,col("bst.EMPLOYEE_ID").cast(DecimalType(6)).alias("EMPLOYEE_ID")
    ,col("bst.TIME_KEY").alias("TIME_KEY")
    ,col("bst.SELLING_PRICE").alias("SELLING_PRICE")
    ,col("sp.PRODUCT_COST").alias("UNIT_COST")
    ,col("bst.DELIVERY_CHARGES").alias("DELIVERY_CHARGES")
    ,col("bst.QUANTITY").alias("SALES_QTY")
    ,col("bst.DISCOUNT").alias("DISCOUNT")
    ,col("bst.HOLDBACK").alias("HOLDBACK")
    ,col("bst.REBATE").alias("REBATE")
)

## Carga

In [7]:
# - Reemplaza la información por particion en la capa silver
df_BroSalTra.write.format("delta") \
      .option("partitionOverwriteMode", "dynamic") \
      .mode("overwrite") \
      .saveAsTable("silver.transactions")

## Vacuum

In [8]:
# - Actualiza la configuración para que pueda establecer un período de retención cero.
#spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

#- Ejecutemos el comando de VACUUM 
#spark.sql(f"VACUUM '{vPathSilv}TRANSACTIONS.delta' RETAIN 0 HOURS").show(truncate=False)

## Cierre de SparkSession

In [9]:
spark.stop()